# The shared-feature counterexample, and noise cannot rescue it

One hidden ReLU unit, squared loss, a risk cap on a minority group's error — but every input
coincides at `x = 1`, so the two groups differ **only in label**, not in features. This is the
cleanest version of the phenomenon in `counter.ipynb`: an open set on which `grad f = grad g = 0`
while `g > 0`, so the Lagrangian gradient is exactly zero for every multiplier and every
penalty-based dual method stalls, permanently infeasible.

Closed forms and all the numbers below are derived in `counter_dl_derivation_shared_feature.md`;
this notebook exists to run the actual `humancompatible.train.dual_optim` code on it and to show
that the trap survives **arbitrarily large bounded-variance gradient noise**, provided the run
starts far enough in.

## The problem

    yhat(x; t) = v * relu(w x + b),   t = (w, b, v)
    f(t) = 0.9 * yhat^2 + 0.1 * (yhat - 1)^2
    g(t) = 0.25 * yhat^2 + 0.75 * (yhat - 1)^2 - 0.25 <= 0

A population of 1000 (`p = 0.1` positive) drives `f`; a 20-point subgroup `R` (`q = 0.75`
positive) drives `g`, capped at `eps = 0.25`. All inputs sit at `x = 1`, so the pre-activation
`s = w + b` is shared by every one of them — the whole problem reduces to a scalar function of
`yhat = v * relu(s)`.

In [ ]:
import torch
from matplotlib import pyplot as plt
from humancompatible.train.dual_optim import ALM, PBM, SSG

# every input is x = 1; a 1000-point population (p=0.1 positive) drives f; a 20-point
# subgroup R (q=0.75 positive, mixed labels) drives g
N_TOT, N_POS = 1000, 100               # p = 0.1
R_SIZE, R_POS = 20, 15                 # q = 0.75
X = torch.ones(N_TOT, 1)
Y = torch.cat([torch.ones(N_POS, 1), torch.zeros(N_TOT - N_POS, 1)])
IS_B = torch.zeros(N_TOT, dtype=torch.bool)
IS_B[:R_POS] = True                            # 15 of the positives -> R
IS_B[N_POS:N_POS + (R_SIZE - R_POS)] = True    # 5 of the negatives -> R
X_B = X[IS_B][:1]                      # one R input, for reporting yhat
KAPPA = 0.25                           # = eps, the risk cap on R

INIT_alive = torch.tensor([1.0, -0.1, 1.0])   # x = 0.9 > 0, all points active, g = -0.04
INIT_dead = torch.tensor([-1.0, -1.0, 1.0])   # deep inside the dead half-plane


def predict(t, x, act=torch.relu):
    return t[2] * act(t[0] * x + t[1])


def objective(t, act=torch.relu):
    return ((predict(t, X, act) - Y) ** 2).mean()


def constraint(t, act=torch.relu):     # g <= 0
    yb = predict(t, X[IS_B], act)
    return (((yb - Y[IS_B]) ** 2).mean() - KAPPA).reshape(1)


for name, t in [("dead half-plane (0, -1)", [0.0, -1.0, 1.0]),
                ("dead half-plane (-2, 1)", [-2.0, 1.0, 1.0]),
                ("global min  (0, 0.1)", [0.0, 0.1, 1.0]),
                ("constrained opt (0, 0.5)", [0.0, 0.5, 1.0]),
                ("init", INIT_alive)]:
    t = t if torch.is_tensor(t) else torch.tensor(t)
    print(f"{name:<27} f = {objective(t):.6f}  g = {constraint(t).item():+.4f}  "
          f"yhat = {predict(t, torch.tensor([[1.0]])).item():.4f}")

print("\nf and g are constant on the whole half-plane {w + b < 0}: 0.1 and 0.5 respectively,")
print("about 11% above the true global minimum of f (0.09) -- the loss curve barely flags the trap.")

## The shape

Everything reduces to a scalar problem in `s = w + b` (fixing `v = 1`):

    F(s) = 0.1                (s <= 0),   F(s) = s^2 - 0.2 x + 0.1   (s > 0)
    G(s) = 0.5                (s <= 0),   G(s) = s^2 - 1.5 x + 0.5   (s > 0)

Both curves have a **downward** kink at `s = 0` (`F'(0-)=0` vs `F'(0+)=-0.2`; `G'(0-)=0` vs
`G'(0+)=-1.5`): adding `rho/2 * s^2` only perturbs the derivative continuously, so no finite `rho`
convexifies either function. The feasible set is `g <= 0 <=> x in [0.5, 1.0]`, and the constrained
optimum sits at `s* = 0.5`, `f* = 0.25`, `lambda* = 1.6` — about 2.8x the unconstrained optimum
`f_min = 0.09` at `s = 0.1`.

In [ ]:
def curve(xx):
    f, g = [], []
    for x in xx:
        t = torch.tensor([0.0, float(x), 1.0])
        f.append(objective(t).item()); g.append(constraint(t).item())
    return f, g

def plain_lagrangian(ff, gg, l = 1., rho=0.):
    LL = []
    alm = ALM(m=1, penalty=rho, augmentation="hpr" if rho > 0 else None, init_duals=l)
    for f, g in zip(ff, gg):
        LL.append(alm.forward(torch.tensor(f), torch.tensor(g)))

    return LL

def pbm_lagrangian(ff, gg, l = 1., p = 1.):
    LL = []
    pbm = PBM(m=1, init_duals=l, init_penalties=p, gamma_annealing=False, penalty_annealing=False)
    for f, g in zip(ff, gg):
        LL.append(pbm.forward(torch.tensor(f), torch.tensor(g)))

    return LL
    


s_grid = torch.linspace(-1.0, 1.5, 601)
F, G = curve(s_grid)
L = plain_lagrangian(F, G)
L_pbm = pbm_lagrangian(F, G)

fig, ax = plt.subplots(figsize=(7, 4))
ax.axvspan(0.5, 1.0, color="tab:green", alpha=0.12, label="feasible set")
ax.axvspan(-1.0, 0.0, color="tab:red", alpha=0.10, label="flat half-plane $x \leq 0$")
ax.plot(s_grid, F, label=r"$f(x)$")
ax.plot(s_grid, G, label=r"$g(x)$")
ax.plot(s_grid, L, label=r"$f(x) + g(x)$")
ax.plot(s_grid, L_pbm, label=r"$\mathcal{L} (\lambda=1, p=1)$")
ax.axhline(0, color="k", lw=0.6)
ax.axvline(0.5, color="k", ls=":", lw=1.0)
# ax.plot([0.0], [0.1], "ko", ms=5, mfc="white", zorder=5, label="dead value")
ax.plot([0.1], [0.09], "k*", ms=10, zorder=5, label=rf"global $\min_x (f) = 0.09$")
ax.plot([0.5], [0.25], "ko", ms=7, mfc="yellow", zorder=5, label=r"feasible $\min_x (f) = 0.25$")
ax.set_xlabel(r"$x = w + b$"); ax.legend(fontsize=8)
# ax.set_title("f(s), g(s) -- both kink downward at x = 0")
plt.tight_layout()
fig.savefig(fname="counter_landscape.pdf")

## The stall

At `t0 = (1, -0.1, 1)`: `s = 0.9`, `g = -0.04 < 0` — the constraint is satisfied, so `lambda_1 = 0`
and the augmented Lagrangian is *exactly* `f` at step 0. There `grad f = 1.6 * (1, 1, 0.9)`, and
since both `d yhat/dw` and `d yhat/db` equal `v`, one SGD step moves `s` by `-3.2 * eta`, so the
unit dies **on the first step** iff

    eta >= 9/32 = 0.28125

Below that threshold there is a second, slower route in: the fixed point `s* = 0.5` destabilises
into a growing oscillation once `eta` exceeds a stability bound (`0.18363` under HPR at `rho = 1`,
`0.18733` under the quadratic augmentation), and one downswing crosses `s = 0`.

In [ ]:
def run(make_dual, eta, iters=400, act=torch.relu, init=INIT_alive, train_v=True,
        lam0=None):
    # one primal-dual run; returns a per-iteration log
    t = torch.nn.Parameter(init.clone())
    opt = torch.optim.SGD([t], lr=eta)
    dual = make_dual()
    if lam0 is not None:
        dual.param_groups[0]["params"][0].fill_(lam0)
    log = []
    for k in range(iters):
        f, c = objective(t, act), constraint(t, act)
        L = dual.forward_update(f, c)
        opt.zero_grad()
        L.backward()
        if not train_v:
            t.grad[2] = 0.0
        log.append(dict(w=t[0].item(), b=t[1].item(), s=(t[0] + t[1]).item(),
                        yhat=predict(t, X_B, act).item(), f=f.item(), g=c.item(),
                        lam=dual.duals.max().item(), gn=t.grad.norm().item()))
        opt.step()
    return log


def trapped(log):
    # stalled at an infeasible point: gradient vanished, constraint still violated
    return log[-1]["gn"] < 1e-8 and log[-1]["g"] > 1e-6


def verdict(log):
    z = log[-1]
    if not (z["gn"] == z["gn"] and abs(z["gn"]) < float("inf")):
        return "DIVERGED"
    return "TRAPPED" if trapped(log) else "ok"


def show(name, log):
    z = log[-1]
    print(f"{name:<26} {verdict(log):<8} "
          f"w+b {log[0]['s']:+.3f} -> {z['s']:+.4f} | yhat_B {z['yhat']:+.4f} | f {z['f']:.5f} | "
          f"g {z['g']:+.5f} | lam {z['lam']:.4f} | |grad L| {z['gn']:.2e}"
          f"{' (exactly 0)' if z['gn'] == 0.0 else ''}")


def alm(lr=0.1, penalty=1.0, **kw):
    return lambda: ALM(m=1, lr=lr, penalty=penalty, is_ineq=True, **kw)

runs = {eta: run(alm(), eta, train_v=False) for eta in (0.10, 0.18, 0.30)}
for eta, log in runs.items():
    show(f"eta = {eta}", log)

## Every dual method sees the same zero

The primal direction of ALM, PBM and SSG all lie in `span{grad f, grad g}`, so all of them see
exactly zero on the dead half-plane, for every multiplier and every penalty.

In [ ]:
show("ALM (quadratic)", run(alm(augmentation="quadratic"), 0.30))
show("ALM (hpr)", run(alm(augmentation="hpr"), 0.30))
show("SSG", run(lambda: SSG(m=1), 0.30))
show("PBM", run(lambda: PBM(m=1, gamma=0.1, gamma_annealing=False,
                            penalty_annealing=False), 0.30))

print("\nsame primal trajectory for any initial multiplier and any penalty:")
for lam0 in (0.0, 1.0, 10.0):
    for rho in (0.1, 1.0, 100.0):
        log = run(alm(penalty=rho), 0.30, lam0=lam0)
        print(f"  lam0={lam0:<5} rho={rho:<6} s_final={log[-1]['s']:+.6f} "
              f"g={log[-1]['g']:+.4f} |grad L|={log[-1]['gn']:.1e}")

## Diminishing step size, long horizon

Same setup, but with a Robbins–Monro step size `alpha_k = eta0 / (k + 1)`
(`sum alpha_k = inf`, `sum alpha_k^2 < inf` — the assumption behind Sec. 9 of the derivation
doc) in place of the constant `eta` above, run for 20,000 iterations on both ALM and PBM. The
first step still uses the full `eta0`, so the same threshold `eta0 > 0.2528` still decides
whether that first move already lands in `D` — decaying the step size afterwards cannot
undo it; once in `D` the gradient is exactly zero, so no schedule can pull it back out
either.

In [ ]:
def run_dimin(make_dual, eta0, iters, act=torch.relu, init=INIT_alive, train_v=True):
    # like run(), but with a Robbins-Monro step size alpha_k = eta0 / (k + 1)
    t = torch.nn.Parameter(init.clone())
    opt = torch.optim.SGD([t], lr=eta0)
    dual = make_dual()
    log = []
    for k in range(iters):
        opt.param_groups[0]["lr"] = eta0 / (k + 1)
        f, c = objective(t, act), constraint(t, act)
        L = dual.forward_update(f, c)
        opt.zero_grad()
        L.backward()
        if not train_v:
            t.grad[2] = 0.0
        log.append(dict(w=t[0].item(), b=t[1].item(), s=(t[0] + t[1]).item(),
                        yhat=predict(t, X_B, act).item(), f=f.item(), g=c.item(),
                        lam=dual.duals.max().item(), gn=t.grad.norm().item()))
        opt.step()
    return log


ITERS_LONG = 2_000
ETAS0 = (0.10, 0.20, 0.30)
methods = {"SSL-ALM": alm(), "SPBM": lambda: PBM(m=1, gamma=0.1, gamma_annealing=False,
                                            penalty_annealing=False)}
runs_dimin = {(name, eta0): run_dimin(make, eta0, ITERS_LONG, train_v=False)
              for name, make in methods.items() for eta0 in ETAS0}
for (name, eta0), log in runs_dimin.items():
    show(f"{name} eta0={eta0} (diminishing)", log)


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
colors = {0.10: "tab:blue", 0.20: "tab:green", 0.30: "tab:red"}
styles = {"SSL-ALM": "-", "SPBM": "--"}
for name in methods:
    for eta0 in ETAS0:
        log = runs_dimin[(name, eta0)]
        axs[0].plot([r["s"] for r in log], color=colors[eta0], linestyle=styles[name],
                    label=f"{name} eta0={eta0}")
        axs[1].plot([r["gn"] for r in log], color=colors[eta0], linestyle=styles[name])

axs[0].axhspan(0.5, 1.0, color="tab:green", alpha=0.12)
axs[0].axhspan(-2.0, 0.0, color="tab:red", alpha=0.10)
axs[0].set_ylabel("s = w + b (the iterate)"); axs[0].set_title("iterate trajectory")
axs[0].legend(fontsize=7, ncol=2)

axs[1].set_yscale("symlog", linthresh=1e-12)
axs[1].set_title("||grad L||")

for ax in axs:
    ax.set_xscale("log"); ax.set_xlabel("iteration (log scale)")
plt.tight_layout()


## Minibatch noise cannot rescue the run either

Section 9 of the derivation doc assumes an abstract, bounded-variance noisy oracle -- the
cells above realized that with synthetic Gaussian noise. Real training instead draws
minibatches from a fixed dataset; does that change anything? A concrete instance, matching
the paper's exact setup: 1000 points (`p=0.1` positive) drive `f`; a 20-point subgroup `R`
(`q=0.75` positive) drives the constraint, capped at `eps=0.25`:

    f(s) = p*([s]+ - 1)^2 + (1-p)*[s]+^2 = s^2 - 0.2 x + 0.1   (s >= 0),   0.1   (s < 0)
    g(s) = q*([s]+ - 1)^2 + (1-q)*[s]+^2 - eps = s^2 - 1.5 x + 0.5   (s >= 0),   0.5   (s < 0)

Both still have the same downward kink at `s = 0`. The only change from the cells above is
*how* the per-step estimate is produced: instead of adding synthetic noise to the true
gradient/constraint, each step draws a fresh minibatch of labels and computes the real
per-batch mean -- no noise is injected anywhere.

In [ ]:
N_TOT, N_POS = 1000, 100          # p = 0.1
R_SIZE, R_POS = 20, 15            # q = 0.75
EPS_MB = 0.25
MOREAU_MU = 1.

import math

Y_full = torch.cat([torch.ones(N_POS), torch.zeros(N_TOT - N_POS)])
Y_R = torch.cat([torch.ones(R_POS), torch.zeros(R_SIZE - R_POS)])


def objective_mb(t, y_batch, act=torch.relu):
    yhat = predict(t, torch.ones_like(y_batch), act)
    return ((yhat - y_batch) ** 2).mean()


def constraint_mb(t, y_batch, act=torch.relu):
    yhat = predict(t, torch.ones_like(y_batch), act)
    return (((yhat - y_batch) ** 2).mean() - EPS_MB).reshape(1)


def run_minibatch(make_dual, alpha0, B, D0, iters=100, v0=1.0, seed=0, decay=True,
                   obj_from_R=False):
    # real minibatch SGD: each step draws B fresh labels from the population (for f) and
    # B fresh labels from R (for g) -- no noise is injected, this is the actual estimator.
    # obj_from_R=True draws f's batch from R too (importance-weighted back to p) instead.
    g = torch.Generator().manual_seed(seed)
    t = torch.tensor([-D0 / 2, -D0 / 2, v0])
    dual = make_dual()
    log = []
    tol = 0
    z = t.detach()
    for k in range(1, iters + 1):
        tp = torch.nn.Parameter(t.clone())
        if obj_from_R:
            y_f = Y_R[torch.randint(0, R_SIZE, (B,), generator=g)]
            f = objective_mb_R(tp, y_f)
        else:
            y_f = Y_full[torch.randint(0, N_TOT, (B,), generator=g)]
            f = objective_mb(tp, y_f)
        y_g = Y_R[torch.randint(0, R_SIZE, (B,), generator=g)]
        c = constraint_mb(tp, y_g)
        tol /= math.sqrt(k)
        if dual == "ssw":
            if c > tol:
                L = c
            else:
                L = f
        else:    
            L = dual.forward(f, c)

        L.backward()
        log.append(dict(s=(t[0] + t[1]).item(), f=f.item(), g=c.item(),
                        lam=dual.duals.max().item() if hasattr(dual, "duals") else None, gn=tp.grad.norm().item()))
        if dual != "ssw":
            dual.update(c)
        alpha_k = (alpha0 / k) if decay else alpha0
        t = t - alpha_k * (tp.grad - MOREAU_MU*torch.abs(t - z))
        z = 0.5*z + 0.5*t
    return log


In [ ]:
# the per-sample/per-batch gradient is exactly zero on D, for any batch, any dual state
g_check = torch.Generator().manual_seed(0)
worst_grad = 0.0
for _ in range(2000):
    B = int(torch.randint(1, 50, (1,), generator=g_check))
    s0 = -float(torch.rand(1, generator=g_check)) * 5 - 1e-3    # any x < 0
    lam0 = float(torch.rand(1, generator=g_check)) * 10
    tp = torch.nn.Parameter(torch.tensor([s0, 0.0, 1.0]))
    y_f = Y_full[torch.randint(0, N_TOT, (B,), generator=g_check)]
    y_g = Y_R[torch.randint(0, R_SIZE, (B,), generator=g_check)]
    dual = ALM(m=1, lr=0.1, penalty=1.0, is_ineq=True, augmentation="hpr")
    dual.duals.data.fill_(lam0)
    L = dual.forward_update(objective_mb(tp, y_f), constraint_mb(tp, y_g))
    L.backward()
    worst_grad = max(worst_grad, tp.grad.abs().max().item())
print(f"max |grad| over 2000 random (batch size, s<0, dual state) draws: {worst_grad:.2e}")
assert worst_grad == 0.0
print("exactly zero every time -- the minibatch never has a direction to act on inside D")


In [ ]:
# the batch's *value* of g (not its gradient) does fluctuate -- check the derived variance
print(f"{'B':>4} {'theory Var':>12} {'empirical Var':>14} {'empirical mean':>15}")
for B in (1, 5, 20):
    gg = torch.Generator().manual_seed(0)
    s_fixed = torch.tensor([-1.0, 0.0, 1.0])
    vals = [constraint_mb(s_fixed, Y_R[torch.randint(0, R_SIZE, (B,), generator=gg)]).item()
            for _ in range(20_000)]
    mean_val = sum(vals) / len(vals)
    emp_var = sum((v - mean_val) ** 2 for v in vals) / len(vals)
    theory_var = 0.75 * 0.25 / B
    print(f"{B:4d} {theory_var:12.5f} {emp_var:14.5f} {mean_val:15.4f}")


So the primal gradient is identically zero regardless of the batch, but the batch's *value*
of `g` genuinely fluctuates (matching the table above) -- and the dual, which only sees that
value, reacts to it. Fix one starting point well inside `D` (`D0=2`, so `s0=-1`) and sweep
the batch size `B in {1, 5, 20}` for both ALM and PBM:

In [ ]:
D0 = 2.0
B_LIST = (1, 5, 20)
colors = {1: "tab:red", 5: "tab:orange", 20: "tab:blue"}
methods = {"SSL-ALM": alm(augmentation=None),
           "SPBM": lambda: PBM(m=1, gamma=0.9, gamma_annealing=False, penalty_annealing=False)}
styles = {"SSL-ALM": "-", "SPBM": "--", "SSw": "-."}

fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
s_vals, n_iters = set(), 0
for B in B_LIST:
    for name, make in methods.items():
        log = run_minibatch(make, alpha0=0.3, B=B, D0=D0, seed=0)
        s_vals.update(round(r["s"], 10) for r in log)
        n_iters += len(log)
        axs[0].plot([r["g"] for r in log], color=colors[B], linestyle=styles[name],
                    label=f"B={B}, {name}")
        axs[1].plot([r["lam"] for r in log], color=colors[B], linestyle=styles[name],
                    label=f"B={B}, {name}")

axs[0].axhline(0, color="k", lw=0.8, ls="--")
axs[0].set_ylabel("per-batch constraint estimate g_k"); axs[0].set_title("raw batch noise")
axs[0].legend(fontsize=7, ncol=2)

axs[1].set_ylabel("dual variable lambda_k"); axs[1].set_title("the dual reacts to the noise")
axs[1].legend(fontsize=7, ncol=2)

for ax in axs:
    ax.set_xlabel("iteration")
plt.tight_layout()

print(f"distinct values of s_k across all {len(B_LIST) * len(methods)} runs, {n_iters} iterations total: {s_vals}")
print("the primal iterate never moves at all -- not 'rarely', not 'with high probability': exactly never.")


`lambda_k` climbs and jitters differently across batch sizes (noisier, more erratic for
`B=1`; smoother for `B=20`, consistent with `Var = q(1-q)/B` shrinking), and ALM vs. PBM
diverge from each other once the dual grows large enough to matter to *their own* update
rule -- but `s_k` is bit-for-bit identical across every one of these runs, printed above.
Unlike the bounded-noise case (Doob's inequality, escape probability `<= sigma_s^2 A2 /
D0^2`), there is no probability left over here: for this exact dataset, minibatch resampling
gives the primal gradient exactly zero variance, so the trap holds with probability exactly
1, for any batch size, any dual method, any starting point strictly inside `D`.

Contrast that with what happens outside `D`: three starting points, `D0=1` (trapped,
`s0=-1`), `D0=0` (exactly the kink, `s0=0`), and `D0=-1` (feasible/active side, `s0=1`),
`B=1`:

In [ ]:
D0_LIST = (1.0, -0.0, -1.0)
colors_d0 = {1.0: "tab:red", 0.0: "black", -1.0: "tab:blue"}
N_SEEDS_ITER = 10

methods = {
    "SSL-ALM": alm(augmentation="hpr"),
    "SPBM": lambda: PBM(m=1, gamma=0.9, gamma_annealing=False, penalty_annealing=False),
    "SSw": lambda: "ssw"
}
B = 20
decay = False

fig, ax = plt.subplots(figsize=(9, 5))
for D0 in D0_LIST:
    for name, make in methods.items():
        n_trapped = 0
        for seed in range(N_SEEDS_ITER):
            log = run_minibatch(make, alpha0=0.1, B=B, D0=D0, seed=seed, decay=decay, iters=200)
            x = [r["s"] for r in log]
            n_trapped += x[-1] < 0
            ax.plot(x, color=colors_d0[D0], linestyle=styles[name], alpha=0.6, linewidth=1,
                    label=fr"$x^0={0-D0:g}$, {name}" if seed == 0 else None)
        if D0 == -1.0:
            print(f"s0=1, {name}: {n_trapped}/{N_SEEDS_ITER} seeds ended up trapped (x < 0)")

ax.axhline(0, color="k", lw=0.8, ls=":")
ax.set_xlabel("iteration"); ax.set_ylabel(r"$x^k = w^k + b^k$")


# ax.plot([ax.get_xlim()[0]], [0.09], "k*", ms=10, zorder=5, clip_on=False)
# ax.plot([0.5], [0.25], "ko", ms=7, mfc="yellow", zorder=5, label=r"feasible $\min_x (f) = 0.25$")

plt.axhline(0.5, color="orange", lw=0.8, ls=":")
# ax.set_title(f"individual iterate trajectories ({N_SEEDS_ITER} seeds each): "
#              f"trapped (S0=-1) vs. boundary (S0=0) vs. active (S0=1), B=1")
ax.legend(fontsize=8, loc="upper right")
plt.tight_layout()
if decay:
    plt.savefig(f"counter_iter_B{B}_decay.pdf")
else:
    plt.savefig(f"counter_iter_B{B}.pdf")


## Sampling the objective from the constrained subgroup, reweighted

The cells above draw `f`'s minibatch from the full population (`Y_full`, rate `p = 0.1`) and
`g`'s from `R` (rate `q = 0.75`) -- two separate draws per step. A different, common design
draws a *single* minibatch from `R` and uses it for both terms (one pass over the constrained
subgroup drives everything) -- but then the raw batch mean of `f` is an unbiased estimator of
`q`, not `p`: biased for the actual population objective. Importance-weight each drawn label
to correct it: for `y ~ Bernoulli(q)`,

    w(1) = p / q,   w(0) = (1 - p) / (1 - q)

gives `E_{y~Bernoulli(q)}[w(y) * (yhat - y)^2] = p*(yhat-1)^2 + (1-p)*yhat^2 = f(yhat)`
exactly, for any `yhat` -- an unbiased estimator of the population objective, drawn entirely
from `R`.

In [ ]:
P_POS, Q_POS = N_POS / N_TOT, R_POS / R_SIZE     # p = 0.1, q = 0.75
W_POS, W_NEG = P_POS / Q_POS, (1 - P_POS) / (1 - Q_POS)
print(f"p={P_POS}, q={Q_POS}  ->  importance weights w(1)={W_POS:.4f}, w(0)={W_NEG:.4f}")


def objective_mb_R(t, y_batch, act=torch.relu):
    # objective's minibatch drawn from R, importance-weighted back to the population rate p
    yhat = predict(t, torch.ones_like(y_batch), act)
    w = torch.where(y_batch == 1, W_POS, W_NEG)
    return (w * (yhat - y_batch) ** 2).mean()


gg = torch.Generator().manual_seed(0)
s_fixed = torch.tensor([-1.0, 0.0, 1.0])          # in D, so yhat = 0 exactly
exact_f = objective(s_fixed).item()
print(f"\n{'B':>4} {'mean':>10} {'exact f':>10} {'theory Var':>12} {'empirical Var':>14} {'full-pop Var':>13}")
for B in (1, 5, 20):
    vals = [objective_mb_R(s_fixed, Y_R[torch.randint(0, R_SIZE, (B,), generator=gg)]).item()
            for _ in range(20_000)]
    mean_val = sum(vals) / len(vals)
    emp_var = sum((v - mean_val) ** 2 for v in vals) / len(vals)
    theory_var = P_POS ** 2 * (1 - Q_POS) / Q_POS / B     # on D, only y=1 draws contribute
    full_pop_var = P_POS * (1 - P_POS) / B                # plain full-population sampling, for comparison
    print(f"{B:4d} {mean_val:10.5f} {exact_f:10.5f} {theory_var:12.6f} {emp_var:14.6f} {full_pop_var:13.6f}")

print("\nunbiased (mean matches exact f), and -- because R oversamples the positives that carry")
print("all the signal on D -- lower variance than plain full-population sampling, not higher.")

In [ ]:
# still exactly zero on D: importance weighting is a per-sample scalar multiplier on a loss
# whose gradient is already exactly zero there, so it stays exactly zero for any weight
g_check2 = torch.Generator().manual_seed(1)
worst_grad2 = 0.0
for _ in range(2000):
    B = int(torch.randint(1, 50, (1,), generator=g_check2))
    s0 = -float(torch.rand(1, generator=g_check2)) * 5 - 1e-3    # any x < 0
    lam0 = float(torch.rand(1, generator=g_check2)) * 10
    tp = torch.nn.Parameter(torch.tensor([s0, 0.0, 1.0]))
    y_f = Y_R[torch.randint(0, R_SIZE, (B,), generator=g_check2)]
    y_g = Y_R[torch.randint(0, R_SIZE, (B,), generator=g_check2)]
    dual = ALM(m=1, lr=0.1, penalty=1.0, is_ineq=True, augmentation="hpr")
    dual.duals.data.fill_(lam0)
    L = dual.forward_update(objective_mb_R(tp, y_f), constraint_mb(tp, y_g))
    L.backward()
    worst_grad2 = max(worst_grad2, tp.grad.abs().max().item())
print(f"max |grad| over 2000 draws, objective AND constraint both sampled from R: {worst_grad2:.2e}")
assert worst_grad2 == 0.0
print("exactly zero every time")

Same `D0`/`B` sweep as before (Sec. "Minibatch noise cannot rescue the run either"), now with
the objective's minibatch also drawn from `R` and importance-weighted:

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
s_vals_R, n_iters_R, runs_R = set(), 0, {}
methods_R = {"SSL-ALM": alm(augmentation=None),
             "SPBM": lambda: PBM(m=1, gamma=0.9, gamma_annealing=False, penalty_annealing=False),
            "SSw": lambda: "ssw"}
for B in B_LIST:
    for name, make in methods_R.items():
        log = run_minibatch(make, alpha0=0.3, B=B, D0=2.0, seed=0, obj_from_R=True)  # D0 fixed explicitly: the D0_LIST sweep above overwrites the global D0
        runs_R[(B, name)] = log
        s_vals_R.update(round(r["s"], 10) for r in log)
        n_iters_R += len(log)
        axs[0].plot([r["f"] for r in log], color=colors[B], linestyle=styles[name],
                    label=f"B={B}, {name}")
        axs[1].plot([r["lam"] for r in log], color=colors[B], linestyle=styles[name],
                    label=f"B={B}, {name}")

axs[0].axhline(exact_f, color="k", lw=0.8, ls="--", label="exact f")
axs[0].set_ylabel("per-batch objective estimate f_k"); axs[0].set_title("raw batch noise (objective, from R)")
axs[0].legend(fontsize=7, ncol=2)

axs[1].set_ylabel("dual variable lambda_k"); axs[1].set_title("the dual still only reacts to g's noise")
axs[1].legend(fontsize=7, ncol=2)

for ax in axs:
    ax.set_xlabel("iteration")
plt.tight_layout()

print(f"distinct values of s_k across all {len(B_LIST) * len(methods_R)} runs, {n_iters_R} iterations total: {s_vals_R}")
print("still never moves -- reweighting the objective's own minibatch changes nothing about the trap.")

And the iterate trajectory itself, alongside `||grad L||`, exactly as in the
diminishing-step-size section above -- this time driven by the reweighted-from-`R` objective,
and (as in the `D0_LIST` sweep further up) from three starting points: inside `D` (trapped),
right on the boundary, and outside `D` (active/feasible side):

## Summary

- The dead half-plane `{w + b < 0}` is an open set on which `grad f = grad g = 0` identically, for
  every dual method and every hyperparameter -- confirmed above with ALM (both augmentations), PBM
  and SSG, and with a closed-form gradient cross-checked against autograd.
- No finite penalty convexifies `f` or `g`: both curves have a downward kink at `s = 0`.
- A diminishing Robbins-Monro step size (`alpha_k = eta0/(k+1)`, no noise at all) changes nothing either: the threshold on `eta0` for the first step to land in `D` is unchanged (it only ever sees the un-decayed `eta0`), and once inside, the exactly-zero gradient means no schedule can shrink the run back out -- confirmed for both ALM and PBM over 20,000 iterations.
- Real minibatch resampling (1000 points, `p=0.1`, driving `f`; a 20-point subgroup `R`, `q=0.75`, driving `g`) is even more decisive than an abstract bounded-noise oracle: the per-batch *gradient* is exactly zero on `D` regardless of batch size or composition -- confirmed over 2000 random (batch, dual-state) draws -- while the per-batch *value* of `g` genuinely fluctuates with the derived variance `q(1-q)/B`. The dual reacts to that noise (ALM and PBM diverge from each other once it grows large); the primal iterate does not move at all, for any `B` or starting point in `D` -- not with high probability, exactly never.
- Drawing the objective's minibatch from `R` too (instead of the full population) and importance-weighting each label by `p/q` or `(1-p)/(1-q)` keeps it exactly unbiased for the population `f` -- and, since `R` oversamples the stratum that carries all the signal on `D`, gives *lower* variance than full-population sampling, not higher. The primal gradient is still exactly zero on `D` regardless (a per-sample importance weight is just a finite scalar multiplying an already-zero per-sample gradient), so the trap is unchanged.